# Diabetes Prediction using Machine Learning
**Dataset:** Pima Indians Diabetes Dataset  
**Algorithm:** Random Forest Classifier  
**Platform:** Google Colab (Free)  

---
### What this project does
We predict whether a patient has diabetes based on 8 health measurements like glucose level, BMI, and age. This is a **binary classification** problem — the output is either `1` (Diabetes) or `0` (No Diabetes).

**Run each cell by pressing `Shift + Enter`**

In [ ]:
import pandas as pd                  # for loading and handling data
import numpy as np                   # for number calculations
import matplotlib.pyplot as plt      # for drawing charts
import seaborn as sns                # for prettier charts

from sklearn.model_selection import train_test_split   # splits data into train/test
from sklearn.ensemble import RandomForestClassifier    # our ML model
from sklearn.metrics import (
    accuracy_score,          # how accurate the model is
    confusion_matrix,        # correct vs wrong predictions
    classification_report    # detailed performance report
)
from sklearn.preprocessing import StandardScaler       # normalises the data

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

## Step 1 — Load the Dataset
We load directly from a public URL. **No download needed!**

In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

column_names = [
    "Pregnancies",        # number of times pregnant
    "Glucose",            # blood sugar level
    "BloodPressure",      # blood pressure (mm Hg)
    "SkinThickness",      # skin fold thickness (mm)
    "Insulin",            # insulin level
    "BMI",                # body mass index
    "DiabetesPedigree",   # family history of diabetes score
    "Age",                # age in years
    "Outcome"             # 1 = diabetes, 0 = no diabetes
]

df = pd.read_csv(url, names=column_names)

print(f"Dataset loaded! Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print()
print(df.head())

## Step 2 — Exploratory Data Analysis (EDA)
EDA means **looking at and understanding the data** before building the model.

In [ ]:
print("=" * 50)
print("DATASET INFORMATION")
print("=" * 50)

print("\nData types:")
print(df.dtypes)

print("\nSummary statistics:")
print(df.describe().round(2))

print("\nMissing values:")
print(df.isnull().sum())

print("\nPatient count by Outcome:")
print(df["Outcome"].value_counts())
print("(0 = No Diabetes,  1 = Has Diabetes)")

### Chart 1 — How many patients have diabetes?

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="Outcome", data=df, palette=["#5DCAA5", "#D85A30"])
plt.title("Diabetic vs Non-Diabetic Patients", fontsize=14, fontweight="bold")
plt.xlabel("Outcome  (0 = No Diabetes,  1 = Diabetes)")
plt.ylabel("Number of Patients")
plt.xticks([0, 1], ["No Diabetes (500)", "Diabetes (268)"])
plt.tight_layout()
plt.savefig("chart1_outcome_count.png", dpi=150)
plt.show()
print("Chart 1 saved!")

### Chart 2 — Distribution of each feature

In [ ]:
features = ["Glucose", "BloodPressure", "BMI", "Age",
            "Pregnancies", "Insulin", "SkinThickness", "DiabetesPedigree"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=20, color="#378ADD", edgecolor="white", alpha=0.85)
    axes[i].set_title(col, fontsize=11, fontweight="bold")
    axes[i].set_ylabel("Count")

plt.suptitle("Distribution of Each Feature", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("chart2_distributions.png", dpi=150)
plt.show()
print("Chart 2 saved!")

### Chart 3 — Correlation Heatmap
Shows how strongly each feature is related to diabetes. Closer to 1.0 = stronger link.

In [ ]:
plt.figure(figsize=(10, 7))
corr = df.corr().round(2)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True)
plt.title("Correlation Heatmap\n(How each feature relates to Outcome)",
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("chart3_heatmap.png", dpi=150)
plt.show()
print("Key insight: Glucose has the highest correlation with Outcome (0.47)")

### Chart 4 — Glucose Level vs Diabetes

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="Outcome", y="Glucose", data=df, palette=["#5DCAA5", "#D85A30"])
plt.title("Glucose Level vs Diabetes Outcome", fontsize=13, fontweight="bold")
plt.xlabel("Outcome  (0 = No Diabetes,  1 = Diabetes)")
plt.ylabel("Glucose Level")
plt.xticks([0, 1], ["No Diabetes", "Diabetes"])
plt.tight_layout()
plt.savefig("chart4_glucose_vs_outcome.png", dpi=150)
plt.show()
print("Diabetic patients clearly have higher glucose levels!")

## Step 3 — Data Preprocessing
### Fix zero values
Columns like Glucose and BMI **cannot be 0 in real life**. These zeros are actually missing values. We replace them with the column average.

In [ ]:
cols_to_fix = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

for col in cols_to_fix:
    zero_count = (df[col] == 0).sum()
    df[col] = df[col].replace(0, df[col].mean())
    print(f"Fixed {zero_count} zero values in '{col}'")

print("\nAll zero values replaced with column averages!")

### Separate features (X) and target (y)

In [ ]:
# X = the 8 inputs the model learns from
# y = the answer (0 or 1) we want to predict
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print(f"Features (X) shape : {X.shape}")   # (768, 8)
print(f"Target   (y) shape : {y.shape}")   # (768,)

### Train / Test Split
We train on **80%** of the data and test on the **remaining 20%**. The model never sees test data during training — this gives an honest accuracy measure.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42,     # same result every time you run
    stratify=y           # equal ratio of 0s and 1s in both sets
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

### Scale the features (Normalisation)
`StandardScaler` makes all features have the same scale so no single column unfairly dominates the model.

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)   # learn scale from training data, then apply
X_test  = scaler.transform(X_test)        # apply SAME scale to test data

print("Features scaled successfully!")

## Step 4 — Train the Random Forest Model
Random Forest = **100 decision trees voting together**. Each tree gives a prediction. The majority vote wins. This makes it much more accurate than a single decision tree.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,    # 100 decision trees
    max_depth=6,         # how deep each tree can grow
    random_state=42
)

model.fit(X_train, y_train)   # this is where the model LEARNS

print("Random Forest model trained successfully!")
print(f"Trees used : {model.n_estimators}")

## Step 5 — Evaluate the Model

In [ ]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 50)
print("       MODEL PERFORMANCE")
print("=" * 50)
print(f"\nAccuracy : {accuracy * 100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred,
      target_names=["No Diabetes", "Diabetes"]))

### Chart 5 — Confusion Matrix
Shows how many predictions were correct and where the model made mistakes.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Diabetes", "Diabetes"],
            yticklabels=["No Diabetes", "Diabetes"],
            linewidths=1)
plt.title("Confusion Matrix\n(Actual vs Predicted)", fontsize=13, fontweight="bold")
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("chart5_confusion_matrix.png", dpi=150)
plt.show()

TP = cm[1][1]; TN = cm[0][0]; FP = cm[0][1]; FN = cm[1][0]
print(f"Correctly predicted No Diabetes : {TN}")
print(f"Correctly predicted Diabetes    : {TP}")
print(f"Wrongly predicted Diabetes      : {FP}")
print(f"Missed actual Diabetes cases    : {FN}")

### Chart 6 — Feature Importance
Which health measurements did the model rely on most?

In [ ]:
feature_names = list(X.columns)
importances   = model.feature_importances_
indices       = np.argsort(importances)[::-1]

bar_colors = ["#378ADD" if importances[i] == max(importances) else "#B5D4F4"
              for i in indices]

plt.figure(figsize=(9, 5))
plt.bar(range(len(feature_names)), importances[indices],
        color=bar_colors, edgecolor="white")
plt.xticks(range(len(feature_names)),
           [feature_names[i] for i in indices],
           rotation=30, ha="right", fontsize=10)
plt.title("Feature Importance\n(Which features matter most to the model)",
          fontsize=13, fontweight="bold")
plt.ylabel("Importance Score")
plt.tight_layout()
plt.savefig("chart6_feature_importance.png", dpi=150)
plt.show()

print(f"Most important feature: {feature_names[indices[0]].upper()}")

## Step 6 — Predict for a New Patient
This is the **real-world use** of the model. A doctor enters patient measurements and the model predicts diabetes risk.

In [ ]:
print("=" * 50)
print("   PREDICTING FOR A NEW PATIENT")
print("=" * 50)

# Change these values to test different patients!
new_patient = pd.DataFrame({
    "Pregnancies":      [3],
    "Glucose":          [120],
    "BloodPressure":    [70],
    "SkinThickness":    [20],
    "Insulin":          [80],
    "BMI":              [28.5],
    "DiabetesPedigree": [0.45],
    "Age":              [30]
})

new_scaled  = scaler.transform(new_patient)
prediction  = model.predict(new_scaled)
probability = model.predict_proba(new_scaled)[0]

print("\nPatient details:")
for col in new_patient.columns:
    print(f"  {col:<20}: {new_patient[col].values[0]}")

result = "DIABETES DETECTED" if prediction[0] == 1 else "NO DIABETES"
print(f"\nPrediction      : {result}")
print(f"Confidence      : {max(probability) * 100:.1f}%")
print(f"Risk of Diabetes: {probability[1] * 100:.1f}%")

## Project Summary

In [ ]:
print("=" * 55)
print("      PROJECT SUMMARY — DIABETES PREDICTION")
print("=" * 55)
print(f"  Dataset       : Pima Indians Diabetes Dataset")
print(f"  Total Samples : 768 patients")
print(f"  Features      : 8 (Glucose, BMI, Age, etc.)")
print(f"  Algorithm     : Random Forest (100 trees)")
print(f"  Train/Test    : 80% / 20%")
print(f"  Accuracy      : {accuracy * 100:.2f}%")
print(f"  Top Feature   : {feature_names[indices[0]]}")
print("=" * 55)